# Random Guide-Selection Strategies

Compares random strategies for selecting guide sets.

`guided_search.py` writes one `results.parquet` per run (a run == one
strategy/config at a single guide-set size `k`; see `config.json`).
`helpers.load_runs` stacks the runs listed in `RUNS` into a single tidy long
DataFrame — one row per leg, tagged with its display `mode` (with ` (k=<k>)`
appended) and carrying `r_<metric>` = metric / that seed's unguided baseline.
Because each run is one `k`, the plots put the individual **seed/goal pairs** on
the x-axis (sorted by value) rather than sweeping `k`; overlay runs at different
`k`'s by listing them in `RUNS`. Every plot is an Altair spec over that frame
(see `plots.py`), so the cells below are a handful of declarative calls.

In [ ]:
import altair as alt
import polars as pl

import helpers as H
import plots as P

# Recessive grids, top legends, the validated categorical palette (see plots.THEME).
alt.theme.register("analysis", enable=True)(lambda: P.THEME)

# Datasets here comfortably exceed Altair's 5k-row default guard; we render inline.
alt.data_transformers.enable("default", max_rows=None)


def ratio_long(df: pl.DataFrame, metrics: list[str]) -> pl.DataFrame:
    """Melt reached legs' `r_<metric>` columns into tidy (metric, ratio) rows.

    Keeps the pair keys (`seed`, `goal`) so `pair_strip` can collapse each pair's
    attempts and rank pairs along x.
    """
    return (
        df.filter(pl.col("reached"))
        .unpivot(
            index=["mode", "seed", "goal", "pair"],
            on=[f"r_{m}" for m in metrics],
            variable_name="metric",
            value_name="ratio",
        )
        .with_columns(pl.col("metric").str.strip_prefix("r_"))
    )

In [ ]:
# Configuration: runs to overlay as (display_name, run-dir pattern). Each run is
# one strategy/config at a single k; its k is folded into the mode label. To
# compare k's, list several single-k runs here — each becomes its own mode.
RUNS = [
    ("no-replacement", "run.7"),
    # ("no-replacement k=10", "run.8"),
    # ("smallest-novel", "run.9"),
]

# Cost metrics carried through the plots. `nodes_per_class` is available in the
# frame too; add it to any plot's metric list to surface it. `memory` is jemalloc
# live-heap bytes (stats.allocated), folded as max(leg, guide) and ratioed against
# the seed baseline.
METRICS = ["iters", "nodes", "classes", "total_time", "memory"]

df, meta = H.load_runs(RUNS)
gr = H.goal_reach(df)

## Cost relative to per-seed baseline

Each reached leg is divided by its own seed's full unguided eqsat cost, so the
reference is a horizontal line at `ratio = 1.0` (guides break even; below 1 is
cheaper than unguided). We plot one point per seed/goal pair (median ratio over
that pair's reached attempts), with pairs sorted along x by their ratio so the
strip reads as a sorted curve — ratios are heavy-tailed, so showing the pairs
directly reveals spread a single band would hide. Per-seed pairing (not one flat
average baseline) keeps a dip below the line from reflecting which seeds are in
the mix rather than whether guides helped.

In [ ]:
P.pair_strip(
    ratio_long(df, METRICS),
    "ratio",
    meta,
    title="Cost relative to per-seed baseline",
    y_title="metric / seed baseline",
    metrics=METRICS,
    columns=2,
)

## Reachability summary

Two panels: the per-pair reach rate (fraction of a pair's attempts that reached),
one point per seed/goal pair sorted within each mode; and a per-mode summary bar
for overall leg reach rate and goal coverage (goals with ≥1 success).

In [ ]:
P.reachability(df, gr, meta)

## Cost distribution by mode

Box plots of reached-leg cost, one facet per metric, a box per mode. Node count
includes the guide egraph (`max(trial_nodes, guide_egraph_nodes)`).

In [ ]:
P.cost_boxplots(df, meta, ["iters", "nodes", "memory"])

## Summary table

In [ ]:
(
    df.group_by("mode")
    .agg(
        pl.col("iters").mean().round(2).alias("avg_iters"),
        pl.col("nodes").mean().round(0).alias("avg_nodes"),
        pl.col("classes").mean().round(0).alias("avg_classes"),
        pl.col("nodes_per_class").mean().round(2).alias("avg_nodes_per_class"),
        pl.col("total_time").mean().round(4).alias("avg_time_s"),
        pl.col("reached").mean().round(3).alias("reached_rate"),
        pl.col("reached").sum().alias("n_reached"),
        pl.len().alias("n_legs"),
    )
    .sort("mode")
)

## Goal-level reachability heatmap

Per-goal reach rate with goals on the y-axis (sorted by average reachability,
hardest at top) and one column per mode.

In [ ]:
P.reach_heatmap(gr, meta)

## Best-guide improvement over baseline (per goal)

For each goal we take the best sampled guide — the minimum of
`metric / seed_baseline` over all attempts that reached it — giving one point per
goal, with goals sorted along x by that best ratio. `ratio = 1.0` (dashed line)
is break-even; below 1 means the best guide is cheaper than unguided. Per-goal
(not per-seed) and unitless.

In [ ]:
P.pair_strip(
    ratio_long(df, ["nodes", "total_time", "memory"]),
    "ratio",
    meta,
    title="Best-guide improvement over baseline (per goal)",
    y_title="best metric / seed baseline",
    metrics=["nodes", "total_time", "memory"],
    group_by=["goal"],  # best (min ratio) guide per goal
    group_reduce="min",
)